# Signet Supply Model version 3

Supposing we know the serial number, we can reduce the dimensionality of the data, as the serial number entails the location (brand), and model. 

### Plan:
1. Reduce data to `month`, `serial number`, `color`, and `line count`
2. Feature engineer lag
3. Impute zeros if needed (sometimes lightGBM does this well on it's own)

## Library

In [1]:
import pypyodbc as podbc
import pandas as pd
from pandas.api.types import CategoricalDtype
import datetime
from datetime import date
import matplotlib.pyplot as plt
# import statsmodels.api as sm
import plotly.express as px

import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from typing import Iterable, List, Optional, Union, Tuple

import seaborn as sns

from optuna.integration import LightGBMPruningCallback
from lightgbm import early_stopping, log_evaluation
import warnings

import joblib

## Data

### Load Data

In [ ]:
# raw = pd.read_excel(r"finalrawdatamodified.xlsx")

In [ ]:
# raw.columns

Index(['Cal Year/Month', 'Serial Number', 'Ship To - Name', 'Model Desc',
       'Total Read', 'Model', 'Total AMV', 'Color', 'Created On', 'Line Count',
       'brand'],
      dtype='object')

In [ ]:
# raw_month_group = raw[['Cal Year/Month', 'Serial Number', 'Ship To - Name', 'Model Desc',
#        'Total Read', 'Model', 'Total AMV', 'Color', 'Line Count',
#        'brand']].groupby(['Cal Year/Month', 'Serial Number', 'Ship To - Name', 'Model Desc',
#        'Total Read', 'Model', 'Color', 
#        'brand']).agg(
#            total_amv = ('Total AMV', 'sum'), 
#            line_count = ("Line Count", 'sum')
#        ).reset_index()

In [ ]:
# raw_month_group

,Cal Year/Month,Serial Number,Ship To - Name,Model Desc,Total Read,Model,Color,brand,total_amv,line_count
0,202409,AAJR011201113,JARED 8386,**BIZHUB C4000I COLOR SFP,14286,C4000I,C,Jared,66630,1
1,202409,AAJR011201113,JARED 8386,**BIZHUB C4000I COLOR SFP,14286,C4000I,K,Jared,66630,0
2,202409,AAJR011201113,JARED 8386,**BIZHUB C4000I COLOR SFP,14286,C4000I,M,Jared,66630,0
3,202409,AAJR011201113,JARED 8386,**BIZHUB C4000I COLOR SFP,14286,C4000I,Y,Jared,66630,1
4,202409,AAJR011201123,JARED 8362,**BIZHUB C4000I COLOR SFP,3553,C4000I,C,Jared,65910,0
...,...,...,...,...,...,...,...,...,...,...
31935,202412,AAJT011206593,JARED 8450,BIZHUB C3300I COLOR SFP,16850,C3300I,Y,Jared,82708,0
31936,202412,AAJT011206604,JARED 9268,BIZHUB C3300I COLOR SFP,8643,C3300I,C,Jared,63054,0
31937,202412,AAJT011206604,JARED 9268,BIZHUB C3300I COLOR SFP,8643,C3300I,K,Jared,63054,1
31938,202412,AAJT011206604,JARED 9268,BIZHUB C3300I COLOR SFP,8643,C3300I,M,Jared,44748,0


In [2]:
# Load utilization data
df_Utilization= pd.read_excel(r"Signet_Summary.xlsx", sheet_name="Report 9", header = 1)
df_Utilization = df_Utilization.drop('Unnamed: 0', axis = 1)

# --- ADD THIS BLOCK ---
# Standardize all column names to prevent KeyErrors
df_Utilization.columns = (
    df_Utilization.columns.str.strip()
    .str.replace(' ', '_')
    .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
)
# --------------------

C:\Users\JSpradlin\AppData\Roaming\Python\Python312\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### Group Data and Isolate needed Columns

In [3]:
df_Utilization['Month']=pd.to_datetime(df_Utilization['Created_On']).dt.month_name()
df_Utilization['Year']=pd.to_datetime(df_Utilization['Created_On']).dt.year

df_grouped = df_Utilization[
    ['Period_month_year', 'Year', 'Month', 'Serial_Number', 'Color', 'Line_Count']
    ].groupby(
        ['Period_month_year', 'Year', 'Month', 'Serial_Number', 'Color']
        ).sum().reset_index()

In [8]:
df_grouped[['Period_month_year', 'Year', 'Month', 'Serial_Number', 'Color', 'Line_Count']]

KeyError: "['Serial_Number'] not in index"

In [13]:
df_grouped_copy = df_grouped.copy()

In [15]:
df_grouped.columns

Index(['Period_month_year', 'Year', 'Month', 'Serial_Number', 'Color',
       'Line_Count'],
      dtype='object')

### Densify Data

In [17]:
# Assumes df_grouped has: Brand, Model_Attr, Ship_To_Name, Color, Period_month_year, Line_Count
df_dense = df_grouped.copy()
df_dense['Period'] = pd.to_datetime(df_dense['Period_month_year'], format="%Y%m").dt.to_period('M')

key_cols = ['Serial_Number', 'Color']
df_dense['first_seen'] = df_dense.groupby(key_cols)['Period'].transform('min')
df_dense['last_seen']  = df_dense.groupby(key_cols)['Period'].transform('max')

def densify(g: pd.DataFrame) -> pd.DataFrame:
    full_idx = pd.period_range(g['first_seen'].iloc[0], g['last_seen'].iloc[0], freq='M')
    g = g.set_index('Period').reindex(full_idx)
    g.index.name = 'Period'
    g = g.reset_index()
    # Re-fill keys
    for c in key_cols:
        g[c] = g[c].ffill().bfill()
    # Zero-fill target where missing
    g['Line_Count'] = g['Line_Count'].fillna(0)
    return g

# 1) Densify and give the result a unique index
dense = (
    df_dense.sort_values(key_cols + ['Period'])
            .groupby(key_cols, group_keys=False)
            .apply(densify)
            .reset_index(drop=True)  # <- ensures unique index for safe assignments
)

# (Optional) remove helper columns if present
for c in ('first_seen', 'last_seen'):
    if c in dense.columns:
        dense = dense.drop(columns=c)

# 2) Features
dense = dense.sort_values(key_cols + ['Period']).reset_index(drop=True)
dense['positive'] = (dense['Line_Count'] > 0).astype('int8')

def months_since_last_positive(s: pd.Series) -> pd.Series:
    # Count consecutive months since last positive (=1); zeros accumulate
    out = np.empty(len(s), dtype=np.int32)
    cnt = 0
    arr = s.to_numpy()
    for i, v in enumerate(arr):
        if v == 1:
            cnt = 0
        else:
            cnt += 1
        out[i] = cnt
    return pd.Series(out, index=s.index)

dense['months_since_last_pos'] = (
    dense.groupby(key_cols)['positive']
         .transform(months_since_last_positive)
)

# Sanity checks
assert dense.index.is_unique
assert dense['months_since_last_pos'].notna().all()


C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_28268\248206357.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(densify)


In [18]:
df_grouped = dense.copy()

In [19]:
df_grouped['Period_month_year'] = pd.to_datetime(df_grouped['Period'].astype(str), format = "%Y-%m").dt.strftime("%Y%m").astype(int)
df_grouped['Year'] = pd.to_datetime(df_grouped['Period'].astype(str), format = "%Y-%m").dt.year
df_grouped['Month'] = pd.to_datetime(df_grouped['Period'].astype(str), format = "%Y-%m").dt.month_name()

## Feature Engineering

### Functions

In [20]:

def _parse_month_date(
    df: pd.DataFrame,
    period_col: Optional[str] = "Period_month_year",
    year_col: Optional[str] = "Year",
    month_col: Optional[str] = "Month",
    date_col_out: str = "date",
) -> pd.DataFrame:
    """
    Adds a monthly datetime column named `date_col_out` (month start) to df by parsing:
    - If `period_col` exists (YYYYMM int/str), uses that.
    - Else uses `year_col` + `month_col` (month full name or number).
    """
    df = df.copy()

    if period_col and period_col in df.columns:
        s = df[period_col].astype(str).str.strip()
        # Keep only digits; handle things like '202112.0'
        s = s.str.replace(r"[^0-9]", "", regex=True)
        year = s.str.slice(0, 4).astype(int)
        month = s.str.slice(4, 6).astype(int)
        df[date_col_out] = pd.to_datetime(
            dict(year=year, month=month, day=1)
        )
    elif (year_col in df.columns) and (month_col in df.columns):
        y = df[year_col].astype(int).astype(str)
        m_raw = df[month_col]

        if pd.api.types.is_numeric_dtype(m_raw) or m_raw.astype(str).str.fullmatch(r"\d{1,2}").all():
            # Month is numeric (1-12)
            m = m_raw.astype(int).clip(1, 12).astype(str).str.zfill(2)
            df[date_col_out] = pd.to_datetime(y + "-" + m + "-01", format="%Y-%m-%d")
        else:
            # Month is a name like 'December'
            df[date_col_out] = pd.to_datetime(y + "-" + m_raw.astype(str), format="%Y-%B")
            df[date_col_out] = df[date_col_out].values.astype("datetime64[M]")  # normalize to month
    else:
        raise ValueError(
            "Could not infer date. Provide either `period_col` or both `year_col` and `month_col`."
        )

    # Normalize to month start
    df[date_col_out] = df[date_col_out].values.astype("datetime64[M]")
    return df


def add_time_lag_features(
    df: pd.DataFrame,
    value_col: str = "Line_Count",
    group_cols: Union[List[str], Tuple[str, ...]] = ("Serial_Number", "Color"),
    period_col: Optional[str] = "Period_month_year",
    year_col: Optional[str] = "Year",
    month_col: Optional[str] = "Month",
    date_col: str = "date",
    lags: Iterable[int] = (1, 3, 6, 12),
    rolling_windows: Iterable[int] = (3, 6, 12),
    agg_func: str = "sum",
    fill_missing: Optional[Union[int, float, str]] = 0,   # 0, "ffill", "bfill", or None (leave NaN)
    ensure_unique: bool = True,
    return_full: bool = True,
) -> pd.DataFrame:
    """
    Create lag and rolling window features for a monthly time series per group.

    Parameters
    ----------
    df : DataFrame
        Input data.
    value_col : str
        Target/metric to lag (e.g., 'Line Count').
    group_cols : list/tuple of str
        Columns defining a series (e.g., Brand, Ship To Name, Color).
    period_col / year_col / month_col : str
        Columns used to derive the monthly date column.
    date_col : str
        Name of the derived datetime (month) column.
    lags : iterable of int
        Lag periods (in months) to compute.
    rolling_windows : iterable of int
        Rolling windows (in months) for stats on *past* values only.
    agg_func : str
        Aggregation across duplicates per (group, date). Typically 'sum' or 'mean'.
    fill_missing : 0, "ffill", "bfill", or None
        How to fill missing months after reindex: zero-fill, forward-fill, back-fill, or leave NaN.
    ensure_unique : bool
        If True, aggregates to one row per (group, month) before feature engineering.
    return_full : bool
        If True, returns the feature-augmented wide frame with group/date/value and features.
        If False, returns only feature columns + keys (suitable for merge).

    Returns
    -------
    DataFrame with lag/rolling/pct-change features.
    """
    if isinstance(group_cols, (list, tuple)):
        group_cols = list(group_cols)
    else:
        group_cols = [group_cols]

    # 1) Parse/attach the monthly date column
    df = _parse_month_date(
        df,
        period_col=period_col,
        year_col=year_col,
        month_col=month_col,
        date_col_out=date_col,
    )

    # 2) Keep relevant columns and ensure value is numeric
    cols_needed = group_cols + [date_col, value_col]
    missing_cols = [c for c in cols_needed if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    work = df[cols_needed].copy()
    work[value_col] = pd.to_numeric(work[value_col], errors="coerce")

    # 3) Aggregate if duplicates per (group, month)
    if ensure_unique:
        work = (
            work.groupby(group_cols + [date_col], dropna=False, as_index=False)
                .agg({value_col: agg_func})
        )

    # 4) Build a complete monthly index per group to avoid irregular gaps
    work = work.sort_values(group_cols + [date_col])
    def _reindex_group(g):
        # full monthly range for this group's min..max date
        full_idx = pd.date_range(g[date_col].min(), g[date_col].max(), freq="MS")
        g = g.set_index(date_col).reindex(full_idx)  # introduces NaNs for gaps
        g.index.name = date_col
        # bring group keys back as columns
        for col in group_cols:
            g[col] = g[col].ffill().bfill().iloc[0] if g[col].isna().any() else g[col]
        # fill value if requested
        if fill_missing is not None:
            if fill_missing == "ffill":
                g[value_col] = g[value_col].ffill()
            elif fill_missing == "bfill":
                g[value_col] = g[value_col].bfill()
            else:
                g[value_col] = g[value_col].fillna(fill_missing)
        return g.reset_index()

    work = (
        work.groupby(group_cols, dropna=False, group_keys=False)
            .apply(_reindex_group)
            .rename(columns={"index": date_col})
    )

    # 5) Compute lags and rolling features (no leakage)
    work = work.sort_values(group_cols + [date_col])

    # Lags
    for L in sorted(set(int(x) for x in lags if x >= 1)):
        work[f"{value_col}_lag{L}"] = (
            work.groupby(group_cols, dropna=False)[value_col].shift(L)
        )

    # Rolling stats on past values only: use lag1 as the base to avoid leakage
    base = work.groupby(group_cols, dropna=False)[value_col].shift(1)
    for W in sorted(set(int(w) for w in rolling_windows if w >= 2)):
        roll = base.groupby(work[group_cols].apply(tuple, axis=1)).rolling(W, min_periods=1)
        # pandas trick: need an index alignment; simpler path:
        work[f"{value_col}_roll{W}_mean"] = (
            work.groupby(group_cols, dropna=False)[value_col]
                .shift(1).rolling(W, min_periods=1).mean()
        )
        work[f"{value_col}_roll{W}_median"] = (
            work.groupby(group_cols, dropna=False)[value_col]
                .shift(1).rolling(W, min_periods=1).median()
        )
        work[f"{value_col}_roll{W}_std"] = (
            work.groupby(group_cols, dropna=False)[value_col]
                .shift(1).rolling(W, min_periods=2).std()
        )
        work[f"{value_col}_roll{W}_min"] = (
            work.groupby(group_cols, dropna=False)[value_col]
                .shift(1).rolling(W, min_periods=1).min()
        )
        work[f"{value_col}_roll{W}_max"] = (
            work.groupby(group_cols, dropna=False)[value_col]
                .shift(1).rolling(W, min_periods=1).max()
        )

    # % change MoM (uses prior value → no leakage)
    work[f"{value_col}_pct_change_1"] = (
        work.groupby(group_cols, dropna=False)[value_col].pct_change(periods=1)
    )

    # YoY change if lag12 requested
    if any(l == 12 for l in lags):
        work[f"{value_col}_pct_change_12"] = (
            work.groupby(group_cols, dropna=False)[value_col].pct_change(periods=12)
        )

    # 6) Return features
    if return_full:
        return work
    else:
        feature_cols = [c for c in work.columns if c not in group_cols + [date_col, value_col]]
        return work[group_cols + [date_col] + feature_cols]


### Call Functions

In [21]:
features = add_time_lag_features(
    df_grouped,
    value_col="Line_Count",
    group_cols=["Serial_Number", "Color"],
    period_col="Period_month_year",
    lags=(1, 3, 6, 12),
    rolling_windows=(3, 6, 12),
    agg_func="sum",
    fill_missing=0,
    ensure_unique=True,
    return_full=True,
)

# Recreate the YYYYMM key from the features' date
features = features.copy()
features["Period_month_year"] = (
    features["date"].dt.year * 100 + features["date"].dt.month
).astype(int)

df_with_features = df_grouped.merge(
    features.drop(columns=["date", 'Line_Count']),
    on=["Serial_Number", "Period_month_year"],
    how="left",
)


C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_28268\2128914970.py:146: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_reindex_group)


### Setting up training and test data

In [22]:
split_date = '2025-01-01'

df_with_features['date'] = pd.to_datetime(df_with_features['Period_month_year'], format = "%Y%m")

target = 'Line_Count'

# Use only the training portion for tuning
train_data = df_with_features[(df_with_features['date'] < split_date) ]
test = df_with_features[df_with_features['date'] >= split_date]

group_features = df_with_features.drop('Line_Count', axis = 1).columns

In [23]:
warnings.filterwarnings("ignore", category=UserWarning)

# ========= 1) INPUTS =========
# Use your prepared dataframe with engineered lags/rollings:
train_data = df_with_features.copy()

# Choose your raw feature wish-list (we will sanitize it below)
group_features = [
    # ------- engineered features (rename to your actual columns) -------
    "Line_Count_lag1", "Line_Count_lag3", "Line_Count_lag6", "Line_Count_lag12",
    "Line_Count_roll3_mean", "Line_Count_roll6_mean", "Line_Count_roll12_mean",
    "Line_Count_pct_change_1", "Line_Count_pct_change_12",
    # ------- raw categoricals (we will set to pandas 'category') -------
    "Brand", 'Model_Attr', "Ship_To_Name", "Color",
]
target = "Line_Count"

# ========= 2) ENSURE MONTHLY DATE COLUMN =========
def ensure_month_date(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Prefer 'date' if present and datetime-like
    if "date" in out.columns and np.issubdtype(out["date"].dtype, np.datetime64):
        out["date"] = out["date"].values.astype("datetime64[M]")
        return out

    # If 'year_month' exists (e.g., '2024-12')
    if "year_month" in out.columns:
        out["date"] = pd.to_datetime(out["year_month"]).values.astype("datetime64[M]")
        return out

    # If 'Period year month' (YYYYMM) exists
    for col in ["Period year month", "Period_month_year"]:
        if col in out.columns:
            s = (
                out[col].astype(str)
                .str.replace(r"[^0-9]", "", regex=True)
                .str.zfill(6)  # YYYYMM
            )
            out["date"] = pd.to_datetime(
                s.str[:4] + "-" + s.str[4:6] + "-01", format="%Y-%m-%d"
            ).values.astype("datetime64[M]")
            return out

    raise ValueError("Provide 'date', 'year_month', or 'Period year month' (YYYYMM).")

train_data = ensure_month_date(train_data)
train_data = train_data.sort_values("date").reset_index(drop=True)

# ========= 3) TIME COVARIATES & FEATURE SANITIZATION =========
# Time covariates
train_data["year_num"] = train_data["date"].dt.year.astype("int16")
train_data["month_num"] = train_data["date"].dt.month.astype("int8")
train_data["month_sin"] = np.sin(2 * np.pi * train_data["month_num"] / 12).astype("float32")
train_data["month_cos"] = np.cos(2 * np.pi * train_data["month_num"] / 12).astype("float32")
train_data["t_idx"] = (train_data["year_num"] * 12 + train_data["month_num"]).astype("int32")

# Add covariates to feature list (no duplicates)
for c in ["year_num", "month_num", "month_sin", "month_cos", "t_idx"]:
    if c not in group_features:
        group_features.append(c)

# Remove any raw datetime or period keys from features if present
for col_to_drop in ["date", "year_month", "Period year month", "Period_month_year"]:
    if col_to_drop in group_features:
        group_features.remove(col_to_drop)

# Ensure target numeric
train_data[target] = pd.to_numeric(train_data[target], errors="coerce")

# Convert string categoricals to pandas category
raw_cat_cols = ["Brand", 'Model_Attr', "Ship_To_Name", "Color"]
raw_cat_cols = [c for c in raw_cat_cols if c in train_data.columns and c in group_features]
for c in raw_cat_cols:
    if train_data[c].dtype == "object":
        train_data[c] = train_data[c].astype("category")
    # Optional: add explicit category for missing
    if str(train_data[c].dtype) == "category" and train_data[c].isna().any():
        train_data[c] = train_data[c].cat.add_categories(["__Unknown__"]).fillna("__Unknown__")

# Keep only columns that actually exist
group_features = [c for c in group_features if c in train_data.columns]

# Drop all-NaN feature columns (can happen with rare lags)
all_nan_cols = [c for c in group_features if train_data[c].isna().all()]
if all_nan_cols:
    group_features = [c for c in group_features if c not in all_nan_cols]

# Build final feature list allowed by LightGBM: numeric or category only
allowed = list(train_data.select_dtypes(include=["number", "category"]).columns)
group_features = [c for c in group_features if c in allowed]

# Recompute cat_cols from the *final* group_features
cat_cols = [c for c in group_features if str(train_data[c].dtype) == "category"]

# Optional: downcast numerics to save memory
num_cols = train_data[group_features].select_dtypes(include=["int64", "float64", "Int64", "Float64"]).columns
for c in num_cols:
    if "float" in str(train_data[c].dtype):
        train_data[c] = train_data[c].astype("float32")
    else:
        train_data[c] = train_data[c].astype("int32")

# Final validation: no object or datetime types in features
bad_obj = train_data[group_features].select_dtypes(include=["object"]).columns.tolist()
bad_dt  = train_data[group_features].select_dtypes(include=["datetime", "datetime"]).columns.tolist()
if bad_obj or bad_dt:
    raise TypeError(f"Remove/convert non-numeric features before training. object: {bad_obj} | datetime: {bad_dt}")

# Also ensure no NaN in y
train_data = train_data[~train_data[target].isna()].reset_index(drop=True)



## Model

In [25]:
train_data.shape

(95489, 37)

### Hyperparameter Tuning and Model Creation

In [26]:
# ========= 4) TIME SERIES CV + OPTUNA =========
tscv = TimeSeriesSplit(n_splits=5, gap=0)  # set gap>0 if you want a buffer

def objective(trial: optuna.trial.Trial) -> float:
    objective_name = trial.suggest_categorical("objective_name", ["regression", "poisson", "tweedie"])
    params = {
        "objective": objective_name,
        "metric": "rmse",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127, step=8),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "min_sum_hessian_in_leaf": 1e-3,
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "feature_fraction_bynode": trial.suggest_float("feature_fraction_bynode", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 0.5),
        "max_bin": trial.suggest_int("max_bin", 63, 255),
        "force_row_wise": True,
        "first_metric_only": True,
        "seed": 42,
        "bagging_seed": 42,
        "feature_fraction_seed": 42,
    }
    if objective_name == "tweedie":
        params["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.9)

    rmse_scores = []

    for i, (tr_idx, va_idx) in enumerate(tscv.split(train_data)):
        tr = train_data.iloc[tr_idx]
        va = train_data.iloc[va_idx]

        X_tr, y_tr = tr[group_features], tr[target]
        X_va, y_va = va[group_features], va[target]

        # Poisson/Tweedie require non-negative labels
        if objective_name in ("poisson", "tweedie"):
            if (y_tr < 0).any() or (y_va < 0).any():
                return float("inf")

        ds_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols if cat_cols else "auto")
        ds_va = lgb.Dataset(X_va, label=y_va, reference=ds_tr, categorical_feature=cat_cols if cat_cols else "auto")

        callbacks = [early_stopping(stopping_rounds=100, verbose=False), log_evaluation(period=0)]
        if i == tscv.get_n_splits() - 1:
            callbacks.append(LightGBMPruningCallback(trial, "rmse"))

        model = lgb.train(
            params=params,
            train_set=ds_tr,
            valid_sets=[ds_va],
            num_boost_round=5000,
            callbacks=callbacks
        )

        preds = model.predict(X_va, num_iteration=model.best_iteration)
        if objective_name in ("poisson", "tweedie"):
            preds = np.clip(preds, 0, None)

        rmse = np.sqrt(mean_squared_error(y_va, preds))
        rmse_scores.append(rmse)

    return float(np.mean(rmse_scores))

study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params.copy()
objective_name = best_params.pop("objective_name", "regression")
best_params.update({
    "objective": objective_name,
    "metric": "rmse",
    "verbosity": -1,
    "force_row_wise": True,
    "first_metric_only": True,
    "seed": 42
})
print("Best Parameters:", best_params)

[I 2025-09-03 15:35:40,876] A new study created in memory with name: no-name-a1b389a5-4ff4-4e91-946d-84e7af4813d9


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-03 15:35:48,678] Trial 0 finished with value: 0.29908301386106356 and parameters: {'objective_name': 'regression', 'learning_rate': 0.07978620548856427, 'num_leaves': 127, 'max_depth': 11, 'min_data_in_leaf': 164, 'feature_fraction': 0.8992665672768878, 'feature_fraction_bynode': 0.988245843438083, 'bagging_fraction': 0.8875534205062527, 'bagging_freq': 7, 'lambda_l1': 5.284689782671289, 'lambda_l2': 0.47202290422688864, 'min_gain_to_split': 0.4357971252595871, 'max_bin': 84}. Best is trial 0 with value: 0.29908301386106356.
[I 2025-09-03 15:36:02,663] Trial 1 finished with value: 0.28170632046243044 and parameters: {'objective_name': 'poisson', 'learning_rate': 0.07284837535050648, 'num_leaves': 79, 'max_depth': 4, 'min_data_in_leaf': 188, 'feature_fraction': 0.7053329594752078, 'feature_fraction_bynode': 0.8428256083251566, 'bagging_fraction': 0.9136736252076396, 'bagging_freq': 2, 'lambda_l1': 0.9216144829232098, 'lambda_l2': 3.623444612595532, 'min_gain_to_split': 0.0919

### Final Evaluation

In [110]:
# ========= 5) FINAL TRAIN / TEST EVALUATION =========
cutoff = pd.Timestamp("2025-01-01")
train_final = train_data[train_data["date"] < cutoff].copy()
test_final  = train_data[train_data["date"] >= cutoff].copy()

X_tr, y_tr = train_final[group_features], train_final[target]
X_te, y_te = test_final[group_features],  test_final[target]

ds_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols if cat_cols else "auto")
ds_te = lgb.Dataset(X_te, label=y_te, reference=ds_tr, categorical_feature=cat_cols if cat_cols else "auto")

model = lgb.train(
    best_params,
    ds_tr,
    valid_sets=[ds_te],
    num_boost_round=5000,
    callbacks=[early_stopping(stopping_rounds=200), log_evaluation(period=100)]
)

preds = model.predict(X_te, num_iteration=model.best_iteration)
if objective_name in ("poisson", "tweedie"):
    preds = np.clip(preds, 0, None)

mae  = mean_absolute_error(y_te, preds)
rmse = np.sqrt(mean_squared_error(y_te, preds))
r2   = r2_score(y_te, preds)

print(f"Final Model MAE:  {mae:.4f}")
print(f"Final Model RMSE: {rmse:.4f}")
print(f"Final Model R^2:  {r2:.4f}")

# Optional: top feature importances
fi = pd.Series(model.feature_importance(importance_type="gain"), index=group_features).sort_values(ascending=False)
print(fi.head(20))


Training until validation scores don't improve for 200 rounds
[100]	valid_0's rmse: 0.272695
[200]	valid_0's rmse: 0.263297
[300]	valid_0's rmse: 0.268087
Early stopping, best iteration is:
[143]	valid_0's rmse: 0.257692
Final Model MAE:  0.0469
Final Model RMSE: 0.2577
Final Model R^2:  0.9704
Line_Count_pct_change_1     175836.358557
Line_Count_lag1              98571.969350
Line_Count_lag3              25096.405518
Line_Count_pct_change_12     21656.238257
Model_Attr                   14119.897530
Line_Count_roll6_mean        12450.640701
Line_Count_roll12_mean       11844.488146
Line_Count_lag12             11781.952574
Line_Count_roll3_mean        11518.413125
Brand                        10943.173207
t_idx                         8075.814844
Color                         4196.990321
Ship_To_Name                  3891.655189
Line_Count_lag6               2180.316125
year_num                       894.939088
month_cos                      690.095482
month_num                      5

### Import Saved Model

In [13]:
model = joblib.load(r"C:\Users\JSpradlin\Documents\GitHub\Signet-Supply\lightGBM_97_percent_v2.joblib")

cat_cols = [c for c in group_features if str(train_data[c].dtype) == "category"]
cutoff = pd.Timestamp("2025-01-01")
train_final = train_data[train_data["date"] < cutoff].copy()
test_final  = train_data[train_data["date"] >= cutoff].copy()

X_tr, y_tr = train_final[group_features], train_final[target]
X_te, y_te = test_final[group_features],  test_final[target]

X_tr, y_tr = train_final[group_features], train_final[target]
X_te, y_te = test_final[group_features],  test_final[target]

ds_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols if cat_cols else "auto")
ds_te = lgb.Dataset(X_te, label=y_te, reference=ds_tr, categorical_feature=cat_cols if cat_cols else "auto")
preds = model.predict(X_te, num_iteration=model.best_iteration)
# if objective_name in ("poisson", "tweedie"):
#     preds = np.clip(preds, 0, None)

mae  = mean_absolute_error(y_te, preds)
rmse = np.sqrt(mean_squared_error(y_te, preds))
r2   = r2_score(y_te, preds)

print(f"Final Model MAE:  {mae:.4f}")
print(f"Final Model RMSE: {rmse:.4f}")
print(f"Final Model R^2:  {r2:.4f}")

# Optional: top feature importances
fi = pd.Series(model.feature_importance(importance_type="gain"), index=group_features).sort_values(ascending=False)
print(fi.head(20))

Final Model MAE:  0.0722
Final Model RMSE: 0.2529
Final Model R^2:  0.9715
Line_Count_pct_change_1     154611.072926
Line_Count_lag1              68291.480561
Line_Count_lag3              30665.017157
Model_Attr                   12547.245588
Line_Count_pct_change_12     12146.397192
Brand                        10507.023072
Line_Count_roll6_mean         8528.598435
Line_Count_roll12_mean        7954.724568
t_idx                         5330.576169
Line_Count_lag12              3884.855448
Color                         3845.170793
Ship_To_Name                  2621.789646
Line_Count_roll3_mean         2260.427735
Line_Count_lag6               2168.745645
year_num                      1688.808813
month_cos                      521.336431
month_num                      453.206581
month_sin                      270.149485
dtype: float64


### Predicted vs Actual

In [14]:
# Plot actual vs predicted

date_style = 'Period_month_year' # 'Created On'

plot_df = test[[date_style, 'Line_Count']].copy()
plot_df['Predicted'] = preds
plot_df = plot_df.rename(columns={'Line_Count': 'Actual'})

plot_df = plot_df.sort_values(date_style)

plot_df_melted = plot_df.melt(id_vars=date_style, value_vars=['Actual', 'Predicted'],
                              var_name='Type', value_name='Line_Count')

plot_df_melted['date'] = pd.to_datetime(plot_df_melted['Period_month_year'], format = "%Y%m")

fig = px.histogram(plot_df_melted, 
                   x = 'date', 
                   y = 'Line_Count', 
                   color = 'Type', 
                   template = 'plotly_dark', 
                   barmode='group',
                   text_auto = True
                   )
fig.update_xaxes(title = 'Month', tickvals = ['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01'], ticktext = ['Jan 2025', 'Feb 2025', 'Mar 2025', 'Apr 2025'])
fig.show()

In [ ]:
# import joblib

# joblib.dump(model, 'lightGBM_97_percent_v2.joblib')

['lightGBM_97_percent_v2.joblib']

## Forcasting

In [15]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from typing import List
import re

def forecast_lgbm_timeseries(
    model: lgb.Booster,
    historical_df: pd.DataFrame,
    horizon: int,
    group_cols: List[str],
    cat_cols: List[str],
    target_col: str = "Line_Count",
    date_col: str = "date",
    objective_name: str = "regression"
) -> pd.DataFrame:
    """
    Generates robust, memory-efficient future forecasts using a trained LightGBM model.

    This function forecasts iteratively: it predicts t+1, adds the prediction
    to the history, and then uses this updated history to predict t+2, and so on.

    Args:
        model (lgb.Booster): The trained LightGBM model object.
        historical_df (pd.DataFrame): The full dataset used for training, including 
                                      the 'date' column and all original features.
        horizon (int): The number of future months to forecast.
        group_cols (List[str]): Columns that define a unique time series 
                                (e.g., ["Brand", "Color"]).
        cat_cols (List[str]): List of categorical feature names used in the model.
        target_col (str): The name of the target variable to be forecasted.
        date_col (str): The name of the datetime column.
        objective_name (str): The objective used for training ('regression', 'poisson', etc.),
                              used for post-prediction clipping.

    Returns:
        pd.DataFrame: A DataFrame containing the future forecasts with columns for
                      date, group identifiers, and the predicted target.
    """
    print(f"🚀 Starting forecast for {horizon} months...")

    ## ========= 1. Setup Phase =========
    feature_cols = model.feature_name()

    # Automatically determine required history length from feature names
    lags = set()
    windows = set()
    lag_pattern = re.compile(f"^{re.escape(target_col)}_lag(\\d+)$")
    roll_pattern = re.compile(f"^{re.escape(target_col)}_roll(\\d+)_.*")
    
    for col in feature_cols:
        lag_match = lag_pattern.match(col)
        roll_match = roll_pattern.match(col)
        if lag_match:
            lags.add(int(lag_match.group(1)))
        if roll_match:
            windows.add(int(roll_match.group(1)))
            
    max_hist_needed = max(list(lags) + list(windows)) if (lags or windows) else 1
    print(f"Required historical context: {max_hist_needed} months.")

    # Get the last date and create the future date range
    last_date = historical_df[date_col].max()
    future_dates = pd.date_range(start=last_date, periods=horizon + 1, freq="MS")[1:]

    # All unique groups we need to forecast for
    unique_groups = historical_df[group_cols].drop_duplicates().reset_index(drop=True)

    # Store category mappings to ensure consistency
    # category_mappings = {c: historical_df[c].cat.categories for c in cat_cols if c in historical_df.columns and pd.api.types.is_categorical_dtype(historical_df[c])}
    

    category_mappings = {c: historical_df[c].cat.categories for c in cat_cols if c in historical_df.columns and isinstance(historical_df[c].dtype, CategoricalDtype)}
    

    # Create a "state" buffer with the most recent history needed for features
    history_cutoff = last_date - pd.DateOffset(months=max_hist_needed)
    history_buffer = historical_df[historical_df[date_col] > history_cutoff].copy()

    ## ========= 2. Iterative Forecasting Loop =========
    forecasts_list = []

    for i, forecast_date in enumerate(future_dates):
        print(f"  - Forecasting for {forecast_date.strftime('%Y-%m')} ({i+1}/{horizon})")
        
        # Create a placeholder DataFrame for the current time step
        step_df = unique_groups.copy()
        step_df[date_col] = forecast_date
        
        # Combine the history buffer with the current step's placeholder
        # This allows us to calculate features using past data
        combined_df = pd.concat([history_buffer, step_df], ignore_index=True)
        combined_df = combined_df.sort_values(group_cols + [date_col]).reset_index(drop=True)

        # --- Recreate features for the current step ---
        # Lags
        for L in sorted(lags):
            combined_df[f"{target_col}_lag{L}"] = combined_df.groupby(group_cols, observed = True)[target_col].shift(L)
        
        # Rolling window features (match the logic from your training script)
        base_series = combined_df.groupby(group_cols, observed = True)[target_col].shift(1)
        for W in sorted(windows):
            rolling_op = base_series.groupby(combined_df[group_cols].apply(tuple, axis=1), observed = True).rolling(W, min_periods=1)
            combined_df[f'{target_col}_roll{W}_mean'] = rolling_op.mean().reset_index(drop=True)
            combined_df[f'{target_col}_roll{W}_median'] = rolling_op.median().reset_index(drop=True)
            combined_df[f'{target_col}_roll{W}_min'] = rolling_op.min().reset_index(drop=True)
            combined_df[f'{target_col}_roll{W}_max'] = rolling_op.max().reset_index(drop=True)
            # Std needs at least 2 periods
            if W > 1:
                combined_df[f'{target_col}_roll{W}_std'] = base_series.groupby(combined_df[group_cols].apply(tuple, axis=1), observed = True).rolling(W, min_periods=2).std().reset_index(drop=True)


        # Percentage change features
        if f"{target_col}_pct_change_1" in feature_cols:
            combined_df[f"{target_col}_pct_change_1"] = combined_df.groupby(group_cols, observed = True)[target_col].pct_change(periods=1)
        if f"{target_col}_pct_change_12" in feature_cols:
            combined_df[f"{target_col}_pct_change_12"] = combined_df.groupby(group_cols, observed = True)[target_col].pct_change(periods=12)

        # Time-based covariates
        combined_df["year_num"] = combined_df[date_col].dt.year
        combined_df["month_num"] = combined_df[date_col].dt.month
        combined_df["month_sin"] = np.sin(2 * np.pi * combined_df["month_num"] / 12)
        combined_df["month_cos"] = np.cos(2 * np.pi * combined_df["month_num"] / 12)
        combined_df["t_idx"] = (combined_df["year_num"] * 12 + combined_df["month_num"])
        
        # Isolate the rows for the current forecast date
        predict_df = combined_df[combined_df[date_col] == forecast_date].copy()
        
        # Ensure categorical dtypes match the model's expectations
        for c in cat_cols:
            if c in predict_df.columns:
                predict_df[c] = pd.Categorical(predict_df[c], categories=category_mappings[c])

        # Predict
        X_predict = predict_df[feature_cols]
        step_preds = model.predict(X_predict, num_iteration=model.best_iteration)
        
        # Post-process (e.g., ensure non-negativity for count data)
        if objective_name in ("poisson", "tweedie"):
            step_preds = np.clip(step_preds, 0, None)
            
        # Add predictions to the DataFrame
        predict_df[target_col] = step_preds
        forecasts_list.append(predict_df)
        
        # Update the history buffer with the new predictions for the next iteration
        history_buffer = pd.concat([history_buffer, predict_df], ignore_index=True)
    
    ## ========= 3. Finalize and Return =========
    df_forecast = pd.concat(forecasts_list, ignore_index=True)
    
    print("✅ Forecast complete!")
    
    return df_forecast[[date_col] + group_cols + [target_col]]

In [ ]:
# Assume 'model' is your trained lgb.Booster object from the end of your script.
# Assume 'train_data' is your final preprocessed DataFrame before splitting.
# 'group_features' and 'cat_cols' are the lists defined in your script.

# The objective name comes from your Optuna study's best trial
objective_from_best_params = model.params.get("objective", "regression")

# Generate a 12-month forecast
future_forecasts = forecast_lgbm_timeseries(
    model=model,
    historical_df=train_data,
    horizon=12,
    group_cols=["Serial_Number", "Color"],
    cat_cols=cat_cols, # The list of categorical columns from your script
    target_col="Line_Count",
    date_col="date",
    objective_name=objective_from_best_params
)



🚀 Starting forecast for 12 months...
Required historical context: 12 months.
  - Forecasting for 2025-05 (1/12)


C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:115: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:117: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



  - Forecasting for 2025-06 (2/12)


C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:115: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:117: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



  - Forecasting for 2025-07 (3/12)


...

  - Forecasting for 2026-03 (11/12)


C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:115: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:117: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



  - Forecasting for 2026-04 (12/12)


C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:115: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_3948\3725761248.py:117: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



✅ Forecast complete!


In [20]:
future_forecasts['Line_Count_int'] = np.floor(future_forecasts['Line_Count'])

In [21]:
future_forecasts

,date,Brand,Model_Attr,Ship_To_Name,Color,Line_Count,Line_Count_int
0,2025-05-01,03765 BANTER BY PIERCING PA,BRMFCL2750,03765 BANTER BY PIERCING PAGODA,K,0.916697,0.0
1,2025-05-01,BANTER BY PIERCING PAGODA,BRMFCL2750,BANTER BY PIERCING PAGODA 670,K,0.825247,0.0
2,2025-05-01,BANTER BY PIERCING PAGODA,BRMFCL2750,BANTER BY PIERCING PAGODA 2454,K,0.822621,0.0
3,2025-05-01,BANTER BY PIERCING PAGODA,BRMFCL2750,BANTER BY PIERCING PAGODA 2511,K,0.813175,0.0
4,2025-05-01,BANTER BY PIERCING PAGODA,BRMFCL2750,BANTER BY PIERCING PAGODA 2549,K,0.822538,0.0
...,...,...,...,...,...,...,...
78451,2026-04-01,ZALES OUTLET,LXMX421,ZALES OUTLET 2864,K,0.878690,0.0
78452,2026-04-01,ZALES OUTLET,LXMX431,ZALES OUTLET 2810,K,0.862970,0.0
78453,2026-04-01,ZALES OUTLET,LXMX431,ZALES OUTLET 2810,Unknown,0.991602,0.0
78454,2026-04-01,ZALES REPAIR SHOP,BRHLL8360CDW,ZALES REPAIR SHOP 8600,Unknown,1.649975,1.0


In [ ]:
df_with_features

,Period,Period_month_year,Year,Month,Color,Model_Attr,Brand,Ship_To_Name,Line_Count,positive,...,Line_Count_roll6_min,Line_Count_roll6_max,Line_Count_roll12_mean,Line_Count_roll12_median,Line_Count_roll12_std,Line_Count_roll12_min,Line_Count_roll12_max,Line_Count_pct_change_1,Line_Count_pct_change_12,date
0,2023-02,202302,2023,February,K,BRMFCL2750,03765 BANTER BY PIERCING PA,03765 BANTER BY PIERCING PAGODA,1.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-02-01
1,2023-03,202303,2023,March,K,BRMFCL2750,03765 BANTER BY PIERCING PA,03765 BANTER BY PIERCING PAGODA,0.0,0,...,1.0,1.0,1.000000,1.0,NaN,1.0,1.0,-1.0,NaN,2023-03-01
2,2023-04,202304,2023,April,K,BRMFCL2750,03765 BANTER BY PIERCING PA,03765 BANTER BY PIERCING PAGODA,0.0,0,...,0.0,1.0,0.500000,0.5,0.707107,0.0,1.0,NaN,NaN,2023-04-01
3,2023-05,202305,2023,May,K,BRMFCL2750,03765 BANTER BY PIERCING PA,03765 BANTER BY PIERCING PAGODA,0.0,0,...,0.0,1.0,0.333333,0.0,0.577350,0.0,1.0,NaN,NaN,2023-05-01
4,2023-06,202306,2023,June,K,BRMFCL2750,03765 BANTER BY PIERCING PA,03765 BANTER BY PIERCING PAGODA,0.0,0,...,0.0,1.0,0.250000,0.0,0.500000,0.0,1.0,NaN,NaN,2023-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73620,2023-09,202309,2023,September,Unknown,BRHLL8360CDW,ZALES REPAIR SHOP,ZALES REPAIR SHOP 8600,0.0,0,...,0.0,0.0,0.666667,0.0,1.556998,0.0,4.0,NaN,NaN,2023-09-01
73621,2023-10,202310,2023,October,Unknown,BRHLL8360CDW,ZALES REPAIR SHOP,ZALES REPAIR SHOP 8600,0.0,0,...,0.0,0.0,0.666667,0.0,1.556998,0.0,4.0,NaN,NaN,2023-10-01
73622,2023-11,202311,2023,November,Unknown,BRHLL8360CDW,ZALES REPAIR SHOP,ZALES REPAIR SHOP 8600,0.0,0,...,0.0,0.0,0.666667,0.0,1.556998,0.0,4.0,NaN,-1.0,2023-11-01
73623,2023-12,202312,2023,December,Unknown,BRHLL8360CDW,ZALES REPAIR SHOP,ZALES REPAIR SHOP 8600,4.0,1,...,0.0,0.0,0.333333,0.0,1.154701,0.0,4.0,inf,inf,2023-12-01


In [22]:
px.histogram(future_forecasts[future_forecasts['Ship_To_Name']=='JARED 417'],
             x = 'date', 
             y = 'Line_Count_int', 
             template = 'plotly_dark',
             color = 'Color',
             text_auto=True, 
             nbins = 12,
             facet_row = 'Model_Attr',
             height = 700,
             ).update_layout(bargap = 0.1)

In [28]:
df_with_features[df_with_features['Ship_To_Name']=='JARED 417']

,Period,Period_month_year,Year,Month,Color,Model_Attr,Brand,Ship_To_Name,Line_Count,positive,...,Line_Count_roll6_min,Line_Count_roll6_max,Line_Count_roll12_mean,Line_Count_roll12_median,Line_Count_roll12_std,Line_Count_roll12_min,Line_Count_roll12_max,Line_Count_pct_change_1,Line_Count_pct_change_12,date
381,2022-07,202207,2022,July,Unknown,BRHLL8360CDW,JARED,JARED 417,4.0,1,...,0.0,4.0,2.600000,4.0,2.408319,0.0,5.0,NaN,NaN,2022-07-01
382,2022-08,202208,2022,August,Unknown,BRHLL8360CDW,JARED,JARED 417,1.0,1,...,0.0,4.0,2.833333,4.0,2.228602,0.0,5.0,-0.750000,NaN,2022-08-01
383,2022-09,202209,2022,September,Unknown,BRHLL8360CDW,JARED,JARED 417,3.0,1,...,0.0,4.0,2.571429,4.0,2.149197,0.0,5.0,2.000000,NaN,2022-09-01
384,2022-10,202210,2022,October,Unknown,BRHLL8360CDW,JARED,JARED 417,1.0,1,...,0.0,4.0,2.625000,3.5,1.995531,0.0,5.0,-0.666667,NaN,2022-10-01
385,2022-11,202211,2022,November,Unknown,BRHLL8360CDW,JARED,JARED 417,4.0,1,...,1.0,4.0,2.444444,3.0,1.943651,0.0,5.0,3.000000,NaN,2022-11-01
655,2022-10,202210,2022,October,K,BRMFCL5900DW,JARED,JARED 417,1.0,1,...,0.0,1.0,0.300000,0.0,0.483046,0.0,1.0,NaN,NaN,2022-10-01
656,2022-11,202211,2022,November,K,BRMFCL5900DW,JARED,JARED 417,0.0,0,...,0.0,1.0,0.300000,0.0,0.483046,0.0,1.0,-1.000000,NaN,2022-11-01
657,2022-12,202212,2022,December,K,BRMFCL5900DW,JARED,JARED 417,0.0,0,...,0.0,1.0,0.300000,0.0,0.483046,0.0,1.0,NaN,NaN,2022-12-01
658,2023-01,202301,2023,January,K,BRMFCL5900DW,JARED,JARED 417,0.0,0,...,0.0,1.0,0.300000,0.0,0.483046,0.0,1.0,NaN,NaN,2023-01-01
659,2023-02,202302,2023,February,K,BRMFCL5900DW,JARED,JARED 417,1.0,1,...,0.0,1.0,0.300000,0.0,0.483046,0.0,1.0,inf,NaN,2023-02-01
